In [ ]:
import os
import cv2
import glob
import json
import shutil
import albumentations as A

# TODO: Update these paths to your dataset locations
image_dir = "./data/predicted_training_data"  # 原始图片路径
output_image_dir = "./data/augmented_dataset_yolo"  # 增强后的图片路径

In [ ]:
def load_yolo_annotations(yolo_txt_path, img_width, img_height):
    """
    从 YOLO 格式的标注文件中加载标注，并转换为 pascal_voc 格式坐标。
    """
    annotations = []
    if os.path.exists(yolo_txt_path):
        with open(yolo_txt_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                class_id, x_center, y_center, w, h = parts
                class_id = int(class_id)
                x_center = float(x_center)
                y_center = float(y_center)
                w = float(w)
                h = float(h)

                # 转换为绝对坐标
                x_min = (x_center - w/2) * img_width
                y_min = (y_center - h/2) * img_height
                x_max = (x_center + w/2) * img_width
                y_max = (y_center + h/2) * img_height

                # 如果出现负值或超出图像范围可在此进行初步修正或过滤
                x_min = max(x_min, 0)
                y_min = max(y_min, 0)
                x_max = min(x_max, img_width - 1)
                y_max = min(y_max, img_height - 1)

                # 仅在边界框有效的情况下加入
                if x_max > x_min and y_max > y_min:
                    annotations.append((class_id, x_min, y_min, x_max, y_max))
    return annotations

def save_yolo_annotations(yolo_txt_path, annotations, img_width, img_height):
    """
    将 Pascal VOC 格式转换回 YOLO 格式并保存。
    """
    with open(yolo_txt_path, 'w') as f:
        for ann in annotations:
            class_id, x_min, y_min, x_max, y_max = ann
            bbox_w = x_max - x_min
            bbox_h = y_max - y_min
            x_center = (x_min + x_max) / 2.0
            y_center = (y_min + y_max) / 2.0

            x_center_norm = x_center / img_width
            y_center_norm = y_center / img_height
            w_norm = bbox_w / img_width
            h_norm = bbox_h / img_height

            # 仅当归一化后依然有效时才写入
            if w_norm > 0 and h_norm > 0:
                f.write(f"{class_id} {x_center_norm:.6f} {y_center_norm:.6f} {w_norm:.6f} {h_norm:.6f}\n")

def clip_bboxes_to_image(bboxes, image_shape):
    h, w = image_shape[:2]
    clipped_bboxes = []
    for bbox in bboxes:
        if len(bbox) != 4:
            continue
        x_min, y_min, x_max, y_max = bbox
        x_min_clipped = max(0, min(x_min, w - 1))
        y_min_clipped = max(0, min(y_min, h - 1))
        x_max_clipped = max(0, min(x_max, w - 1))
        y_max_clipped = max(0, min(y_max, h - 1))

        if x_max_clipped > x_min_clipped and y_max_clipped > y_min_clipped:
            clipped_bboxes.append([x_min_clipped, y_min_clipped, x_max_clipped, y_max_clipped])
    return clipped_bboxes

def augment_yolo_dataset(image_dir, output_image_dir, augmenter, num_augments=50):
    os.makedirs(output_image_dir, exist_ok=True)

    # 假设原图像为.png
    image_paths = glob.glob(os.path.join(image_dir, "*.png"))
    for img_path in image_paths:
        img = cv2.imread(img_path)
        if img is None:
            continue
        height, width = img.shape[:2]

        base_name = os.path.splitext(os.path.basename(img_path))[0]
        anno_path = os.path.join(image_dir, "predicted", f"predict{int(base_name)}", "labels", base_name + ".txt")

        # 加载并转换标注
        annotations = load_yolo_annotations(anno_path, width, height)

        # 分离出坐标和类别ID
        if len(annotations) == 0:
            annotations_pvoc = []
            category_ids = []
        else:
            # 只保留坐标信息
            annotations_pvoc = [[x_min, y_min, x_max, y_max] for (_, x_min, y_min, x_max, y_max) in annotations]
            category_ids = [cid for (cid, _, _, _, _) in annotations]

        # 在增强前打印检查
        print(f"Image: {img_path}")
        print("Before augmentation:")
        print("annotations_pvoc:", annotations_pvoc)
        print("category_ids:", category_ids)

        for i in range(num_augments):
            if len(annotations_pvoc) > 0:
                # 确保传入augmenter的bboxes全部为4个数的列表
                # 且不包含class_id
                bboxes_for_aug = []
                for bbox in annotations_pvoc:
                    if len(bbox) == 4:
                        bboxes_for_aug.append(bbox)

                transformed = augmenter(image=img, bboxes=bboxes_for_aug, category_ids=category_ids)
                transformed_bboxes = transformed['bboxes']
                transformed_category_ids = transformed['category_ids']

                # 增强后裁剪一次，防止出现负值坐标
                transformed_bboxes = clip_bboxes_to_image(transformed_bboxes, transformed['image'].shape)
            else:
                transformed = augmenter(image=img, bboxes=[], category_ids=[])
                transformed_bboxes = []
                transformed_category_ids = []

            new_image = transformed['image']

            # 在增强后的bbox再次打印检查
            print(f"After augmentation {i}:")
            print("transformed_bboxes:", transformed_bboxes)
            print("transformed_category_ids:", transformed_category_ids)

            # 若没有有效的bbox，则不保存txt文件或保存空的txt文件
            new_annotations = []
            for cid, bbox in zip(transformed_category_ids, transformed_bboxes):
                x_min, y_min, x_max, y_max = bbox
                new_annotations.append((cid, x_min, y_min, x_max, y_max))

            # 保存增强后的图片和标签
            new_file_name = f"{base_name}_{i}.png"
            new_image_path = os.path.join(output_image_dir, new_file_name)
            cv2.imwrite(new_image_path, new_image)

            new_anno_path = os.path.join(output_image_dir, f"{base_name}_{i}.txt")
            save_yolo_annotations(new_anno_path, new_annotations, new_image.shape[1], new_image.shape[0])


def clip_bboxes_to_image(bboxes, image_shape):
    h, w = image_shape[:2]
    clipped_bboxes = []
    for bbox in bboxes:
        x_min, y_min, x_max, y_max = bbox[:4]
        # 将坐标限制在 [0, w-1] 和 [0, h-1] 范围内
        x_min_clipped = max(0, min(x_min, w - 1))
        y_min_clipped = max(0, min(y_min, h - 1))
        x_max_clipped = max(0, min(x_max, w - 1))
        y_max_clipped = max(0, min(y_max, h - 1))
        
        # 确保 x_max > x_min, y_max > y_min，可根据需要筛选掉无效框
        if x_max_clipped > x_min_clipped and y_max_clipped > y_min_clipped:
            # 如果你的框中没有类别ID等额外信息，直接附加4个值即可
            # 如果有其他值（如类别ID），请在此处理
            clipped_bboxes.append([x_min_clipped, y_min_clipped, x_max_clipped, y_max_clipped])
    
    return clipped_bboxes


In [ ]:
augmenter = A.Compose([
    # 拉伸和缩放
    A.Affine(scale=(0.8, 1.2), shear=(10, 10), p=0.5),  # 仿射变换，随机缩放和倾斜
    A.Resize(640, 640, p=0.2),  # 将图片调整为固定大小
    A.RandomResizedCrop(height=640, width=640, scale=(0.7, 1.0), ratio=(0.75, 1.333), p=0.5),  # 随机裁剪后缩放

    # 旋转和翻转
    A.Rotate(limit=60, border_mode=0, p=0.7),  # 随机旋转 ±30°，填充超出部分
    A.HorizontalFlip(p=0.5),  # 随机水平翻转

    # 随机仿射变换
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, border_mode=0, p=0.5),  # 随机平移、缩放、旋转

    # 色彩和模糊
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),  # 随机亮度和对比度调整
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.3),  # 色调和饱和度调整
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),  # 随机高斯模糊

    # 随机噪声
    A.ISONoise(p=0.3),  # 模拟 ISO 噪声
    A.MultiplicativeNoise(multiplier=(0.9, 1.1), per_channel=True, p=0.3),  # 随机乘性噪声

    # 随机遮挡
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.3),  # 随机遮挡部分区域
], bbox_params=A.BboxParams(
    format='pascal_voc', 
    label_fields=['category_ids'],
    min_area=1,
    min_visibility=0.0
))

# 执行增强
augment_yolo_dataset(image_dir, output_image_dir, augmenter)